In [2]:
from importlib.metadata import version

print(f"openai version:  {version("openai")}")
print(f"python-dotenv version:  {version("python-dotenv")}")

openai version:  2.7.1
python-dotenv version:  1.2.1


In [2]:
import os
import re
import pickle
import random
import time
import threading
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from dotenv import load_dotenv
from openai import OpenAI

random.seed(42)

In [3]:
! head -3 data/baike_qa/baike_qa_valid.json

{"qid": "qid_1815059893214501395", "category": "烦恼-恋爱", "title": "请问深入骨髓地喜欢一个人怎么办我不能确定对方是不是喜欢我，我却想 ", "desc": "我不能确定对方是不是喜欢我，我却想分分秒秒跟他在一起，有谁能告诉我如何能想他少一点", "answer": "一定要告诉他你很喜欢他 很爱他!!  虽然不知道你和他现在的关系是什么！但如果真的觉得很喜欢就向他表白啊！！起码你努力过了！  女生主动多少占一点优势的！！呵呵  只愿曾经拥有！  到以后就算感情没现在这么强烈了也不会觉得遗憾啊~！  与其每天那么痛苦的想他 恋他 还不如直接告诉他 ！  不要怕回破坏你们现有的感情！因为如果不告诉他  你可能回后悔一辈子！！  "}
{"qid": "qid_2063849676113062517", "category": "游戏-完美游戏-诛仙", "title": "我登陆诛仙2时总说我账号密码错误，但是我打的是正确的，就算不对我? ", "desc": "", "answer": "被盗号了~我的号在22号那天被盗了，跟你一样情况，link密码与账号错误，我密保都有了呐，邮箱换密码也不行，还被删了号，伤心兼郁闷，呵呵，盗号了。建议跟完美申请把号要回来，或者玩新的号！"}
{"qid": "qid_6625582808814915192", "category": "游戏-网络游戏", "title": "斩魔仙者称号怎么得来的 ", "desc": "斩魔仙者称号怎么得来的", "answer": "楼主您好，以下为转载：\r\r圣诞前热身 来《生肖传说》做斩魔仙者\r\r　　一年一度的圣诞节快要来临了，大街小巷商户们都在忙着准备12月25日圣诞的来临。而这时候，一些妖魔也正蠢蠢欲动准备作乱。作为生肖世界肩负维护世界和平、拯救全人类的生肖使者，怎么能不有所行动，为了生肖世界的安定而做防范准备？！\r\r　　要让妖魔鬼怪能对你有所心悸，除了自己本身武艺要高强，最好能在妖魔界打出知名度，这样，当你的亲朋好友被妖魔袭击时，只要爆出你的名号，这些妖魔上就会落荒而逃，岂不好哉？那么，“斩魔仙者”这个响亮的称号应该足够能震慑住妖魔，让他们铭记在心了吧！\r\r斩魔仙者的称号\r\r　　而且，这个“斩魔仙者”的称

In [4]:
QA_QUALITY_PROMPT_TPL = """
你是一个自然语言处理的专家，你的任务是根据输入的文本来判断文本的质量。

以下是一些低质量文本的判断标准：
1.语句不通：语句严重不通顺且没有意义的话
2.特殊字符：标点符号，破折号，特殊字符，emoji，表情符号占比30%以上
3.广告营销：明显的广告推销，博彩，营销内容
4.低俗内容：色情，暴力，仇恨言论
5.隐私泄露：个人的敏感信息，联系方式等

请根据上面的判断标准输出你的结果，如果是低质量，就输出“Y”，否则输出“N”。
请直接输出最终的结果Y或者N，不要输出理由或者其他无关内容。

输入：{}
"""

# 定义 OpenAI Client

In [5]:
load_dotenv()

llm_client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=os.environ["BASE_URL"]
)

# 写调用逻辑

In [6]:
def chat(prompt, max_retry=3):
    def do_chat(text):
        prompt = QA_QUALITY_PROMPT_TPL.format(text)
        completions = llm_client.chat.completions.create(
            model="deepseek-v3-250324",
            messages=[
                {"role": "system", "content": "你是有用的人工智能助手。"},
                {"role": "user", "content": prompt}
            ]
        )
        return text, completions.choices[0].message.content

    while max_retry > 0:
        try:
            return do_chat(prompt)
        except Exception as e:
            max_retry -= 1
            sleep_seconds = random.randint(1, 4)
            time.sleep(sleep_seconds)
    return  

# 主代码

In [ ]:
MAX_WORKERS = 300
test_sentences = []
fd = open("../data/baike_qa/baike_qa_valid.json")
for idx, line in enumerate(fd):
    info = json.loads(line)
    test_sentences.append((idx, info["title"] + info["answer"]))

fw = open("../data/output_quality_full.json", "w")
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {idx: executor.submit(chat, text) for idx, text in test_sentences}
    for idx in tqdm(futures):
        future = futures[idx]
        result = future.result()
        if result is None:
            continue
        text, spam = result
        info = {"text": text, "spam": spam}
        # print(info)
        fw.write(json.dumps(info, ensure_ascii=False) + "\n")

In [7]:
! wc -l data/output_quality_full.json

44972 data/output_quality_full.json


In [8]:
!head -10 data/output_quality_full.json

{"text": "请问深入骨髓地喜欢一个人怎么办我不能确定对方是不是喜欢我，我却想 一定要告诉他你很喜欢他 很爱他!!  虽然不知道你和他现在的关系是什么！但如果真的觉得很喜欢就向他表白啊！！起码你努力过了！  女生主动多少占一点优势的！！呵呵  只愿曾经拥有！  到以后就算感情没现在这么强烈了也不会觉得遗憾啊~！  与其每天那么痛苦的想他 恋他 还不如直接告诉他 ！  不要怕回破坏你们现有的感情！因为如果不告诉他  你可能回后悔一辈子！！  ", "spam": "N"}
{"text": "我登陆诛仙2时总说我账号密码错误，但是我打的是正确的，就算不对我? 被盗号了~我的号在22号那天被盗了，跟你一样情况，link密码与账号错误，我密保都有了呐，邮箱换密码也不行，还被删了号，伤心兼郁闷，呵呵，盗号了。建议跟完美申请把号要回来，或者玩新的号！", "spam": "Y"}
{"text": "斩魔仙者称号怎么得来的 楼主您好，以下为转载：\r\r圣诞前热身 来《生肖传说》做斩魔仙者\r\r　　一年一度的圣诞节快要来临了，大街小巷商户们都在忙着准备12月25日圣诞的来临。而这时候，一些妖魔也正蠢蠢欲动准备作乱。作为生肖世界肩负维护世界和平、拯救全人类的生肖使者，怎么能不有所行动，为了生肖世界的安定而做防范准备？！\r\r　　要让妖魔鬼怪能对你有所心悸，除了自己本身武艺要高强，最好能在妖魔界打出知名度，这样，当你的亲朋好友被妖魔袭击时，只要爆出你的名号，这些妖魔上就会落荒而逃，岂不好哉？那么，“斩魔仙者”这个响亮的称号应该足够能震慑住妖魔，让他们铭记在心了吧！\r\r斩魔仙者的称号\r\r　　而且，这个“斩魔仙者”的称号并不是人人都能得到的。只有成功挑战70级副本中的隐藏BOSS“羽翼仙”的人才能获得此称号！并且前提条件是在12月18日~12月25日之间第一队成功挑战羽翼仙的人才能获此称号！因此，此称号在全服范围内，是绝对不可能超过5个的！\r\r　　要挑战羽翼仙可不是一件容易的事。首先，要在70级副本中打败4个强大的BOSS！在打完副本的第4个BOSS有一定几率获得道具“羽翼真元”，有了羽翼真元后就可以与羽翼仙进行一场战斗。羽翼仙就站在第4个BOSS的旁边，只是没有道具是不能进入战斗的。\r\r羽翼仙\r\r　　在12月18日~12月25日活动期间成功挑战羽翼仙后